In [ ]:
import pandas as pd
import janitor
import numpy as np
import botometer
import time
import yaml
from tqdm.notebook import tqdm

FP_IN = "../3-parseGeolocation/output/parsed-geoloc.csv"

In [ ]:
with open("secrets.yml", "r") as f:
    secrets = yaml.safe_load(f)

rapidapi_key = secrets["rapidapi_key"]

bomx = botometer.BotometerX(rapidapi_key=rapidapi_key)

**Rate limts**
* each request can check up to 100 accounts
* Prefer user ID, which I have
* Pro plan = 50 requests a day

- (https://rapidapi.com/OSoMe/api/botometer-pro)
- https://botometer.osome.iu.edu/faq#faq-q6

In [ ]:
if False:
    df = (
        pd.read_csv("../3-parseGeolocation/output/parsed-geoloc.csv")
        .clean_names()
        .dropna(subset=["primary_county"])
        .rename_column("author_id", "user_id")
        .drop_duplicates("user_id", ignore_index=True)
        .filter(["user_id"])
        .assign(
            bot_score=np.nan,
            timestamp=np.nan,
            username=np.nan,
        )
    )
    display(df)

In [ ]:
if True:
    df = pd.read_csv("intermediate-backup.csv")
    display(df)

**quota usage:** https://rapidapi.com/console/11005098/billing/subscriptions-and-usage

In [ ]:
if 'query_failed' not in df.columns:
    df['query_failed'] = False

for _ in tqdm(range(1, 50)):
    outstanding = df[df["bot_score"].isna() & ~df["query_failed"]]
    
    if outstanding.empty:
        print("✅ all done, nothing left to query")
        break

    ids_to_query = outstanding["user_id"].head(100).tolist()
    payload = bomx.get_botscores_in_batch(user_ids=ids_to_query)
    
    if payload == []:
        # ALL IDs in batch failed
        df.loc[df["user_id"].isin(ids_to_query), "query_failed"] = True
        print(f"⚠️ Empty payload for batch of {len(ids_to_query)} IDs - all marked as failed")
        continuez
    
    # Check which IDs actually returned results
    returned_ids = [item['user_id'] for item in payload]
    returned_ids_int = [int(uid) for uid in returned_ids]
    
    # Mark IDs that were queried but didn't return results as failed
    missing_ids = set(ids_to_query) - set(returned_ids_int)
    if missing_ids:
        df.loc[df["user_id"].isin(missing_ids), "query_failed"] = True
        print(f"⚠️ {len(missing_ids)} IDs returned no results - marked as failed")
    
    df_payload = pd.DataFrame(payload).assign(user_id=lambda df_: df_["user_id"].astype(int))
    
    # merge new scores
    df = df.merge(df_payload, on="user_id", how="left", validate="1:1", suffixes=("", "_new"))
    
    for col in ["bot_score", "timestamp", "username"]:
        df[col] = df[col].fillna(df[f"{col}_new"])
        df.drop(columns=[f"{col}_new"], inplace=True)
    
    df.to_csv("intermediate-backup.csv", index=False)
    time.sleep(0.6)

In [ ]:
df.to_csv("out/bot-classification.csv", index=False)

In [ ]:
df.query("query_failed==True")